Mount Drive, set CONFIG, and check GPU

In [ ]:
from pathlib import Path
import torch
from google.colab import drive

# Keep all paths and settings in one place so nothing is hardcoded later.
CONFIG = {
    "project_dir": Path("/content/drive/MyDrive/doculens_lava"),
    "data_dir": Path("/content/lava_data"),
    "output_dir": Path("/content/drive/MyDrive/doculens_lava/outputs"),
    "sample_pdf_count": 30,
    "min_text_chars": 50,
}


def setup_workspace(config):
    """
    Input: CONFIG dictionary containing project paths.
    Output: Mounts Drive (falling back to local storage if mounting fails), creates folders, and prints GPU status.
    """
    try:
        # Mount Drive so saved outputs are not lost when Colab disconnects.
        print("Attempting to mount Google Drive...")
        drive.mount("/content/drive")
        print("Google Drive mounted successfully.")
    except Exception as e:
        print(f"\n[WARNING] Drive mount failed: {e}")
        print("Falling back to local runtime storage (data will be lost if runtime disconnects).\n")
        # Fallback to local paths to avoid crash
        config["project_dir"] = Path("/content/doculens_lava")
        config["output_dir"] = Path("/content/doculens_lava/outputs")

    # Create our working directories so later cells never crash on missing paths.
    config["project_dir"].mkdir(parents=True, exist_ok=True)
    config["data_dir"].mkdir(parents=True, exist_ok=True)
    config["output_dir"].mkdir(parents=True, exist_ok=True)

    gpu_available = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if gpu_available else "None (CPU only)"

    print("Project folder ready:", config["project_dir"])
    print("Local data folder ready:", config["data_dir"])
    print("GPU available:", gpu_available, "| Device:", gpu_name)


setup_workspace(CONFIG)

Attempting to mount Google Drive...
Mounted at /content/drive
Google Drive mounted successfully.
Project folder ready: /content/drive/MyDrive/doculens_lava
Local data folder ready: /content/lava_data
GPU available: True | Device: Tesla T4


Download and unzip the dataset with help of kaggle.json

In [ ]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path


def setup_kaggle_key(project_dir):
    """
    Input: Path to your Google Drive project folder containing kaggle.json.
    Output: Copies kaggle.json to ~/.kaggle with safe permissions.
    """
    source_key = project_dir / "kaggle.json"
    kaggle_folder = Path.home() / ".kaggle"
    target_key = kaggle_folder / "kaggle.json"

    if not source_key.exists():
        raise FileNotFoundError("Upload kaggle.json to MyDrive/doculens_lava/ first.")

    # Kaggle CLI requires the key inside ~/.kaggle with private read/write permissions.
    kaggle_folder.mkdir(exist_ok=True)
    shutil.copy(source_key, target_key)
    os.chmod(target_key, 0o600)
    print("Kaggle API key is ready.")


def download_and_unzip_data(competition_name, data_dir):
    """
    Input: Kaggle competition slug and target folder path.
    Output: Downloads the competition zip, extracts it, and prints the top-level files.
    """
    zip_path = data_dir / f"{competition_name}.zip"

    # Skip downloading again if the files are already unzipped in this session.
    if not (data_dir / "train.csv").exists():
        subprocess.run(
            ["kaggle", "competitions", "download", "-c", competition_name, "-p", str(data_dir)],
            check=True,
        )
        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(data_dir)
        zip_path.unlink()  # Delete the zip to save Colab disk space.

    print("Files inside data_dir:")
    for item in sorted(data_dir.iterdir()):
        print(" -", item.name)


setup_kaggle_key(CONFIG["project_dir"])
download_and_unzip_data("lava-challenge-2026", CONFIG["data_dir"])

Kaggle API key is ready.
Files inside data_dir:
 - sample_submission.csv
 - test.csv
 - test_pdfs
 - train.csv
 - train_pdfs


Cell 3: Audit PDFs with PyMuPDF

working in cell 3:

Scans all 5 training PDFs plus 25 random test PDFs. It counts how many pages are nearly blank (to decide if we need OCR) and how many contain images or heavy drawings (to decide if we need a VLM). It also prints text snippets from the 5 training PDFs so we can catch garbled Japanese or Vietnamese encoding immediately.


In [ ]:
!pip install -q pymupdf

import random
import fitz
import pandas as pd


def audit_single_pdf(pdf_path, min_chars):
    """
    Input: Path to one PDF file and minimum character threshold.
    Output: Dictionary with page count, low-text count, visual count, and a text sample.
    """
    document = fitz.open(pdf_path)
    low_text_pages = 0
    visual_pages = 0
    sample_text = ""

    for page in document:
        page_text = page.get_text("text").strip()
        if len(page_text) < min_chars:
            low_text_pages = low_text_pages + 1
        # Images or many vector drawings usually mean charts, diagrams, or complex tables.
        if len(page.get_images()) > 0 or len(page.get_drawings()) > 20:
            visual_pages = visual_pages + 1
        if sample_text == "" and len(page_text) >= min_chars:
            sample_text = page_text[:120].replace("\n", " ")

    page_count = len(document)
    document.close()
    return {
        "file_name": pdf_path.name,
        "page_count": page_count,
        "low_text_pages": low_text_pages,
        "visual_pages": visual_pages,
        "sample_text": sample_text,
    }


def print_audit_summary(audit_table, min_chars):
    """
    Input: DataFrame of per-PDF audit results and minimum character threshold.
    Output: Prints page counts and percentages for the OCR and VLM gates.
    """
    total_pages = audit_table["page_count"].sum()
    low_text_total = audit_table["low_text_pages"].sum()
    visual_total = audit_table["visual_pages"].sum()

    low_text_pct = (low_text_total / max(1, total_pages)) * 100
    visual_pct = (visual_total / max(1, total_pages)) * 100

    print("Audited PDFs:", len(audit_table), "| Total pages:", total_pages)
    print(
        "Pages per PDF -> min:", audit_table["page_count"].min(),
        "| median:", audit_table["page_count"].median(),
        "| max:", audit_table["page_count"].max(),
    )
    print(f"Low-text pages (<{min_chars} chars, OCR gate): {low_text_total} ({low_text_pct:.1f}%)")
    print(f"Visual pages (images/drawings, VLM gate): {visual_total} ({visual_pct:.1f}%)")


train_pdfs = sorted((CONFIG["data_dir"] / "train_pdfs").glob("*.pdf"))
test_pdfs = sorted((CONFIG["data_dir"] / "test_pdfs").glob("*.pdf"))
print("Total train PDFs:", len(train_pdfs), "| Total test PDFs:", len(test_pdfs))

# Use a fixed seed so we audit the exact same random sample of test PDFs every time.
random.seed(42)
sample_test_pdfs = random.sample(test_pdfs, min(25, len(test_pdfs)))

rows = []
for pdf_path in train_pdfs + sample_test_pdfs:
    rows.append(audit_single_pdf(pdf_path, CONFIG["min_text_chars"]))

audit_table = pd.DataFrame(rows)
print_audit_summary(audit_table, CONFIG["min_text_chars"])

# Check real extracted text from the 5 train PDFs to spot broken fonts or encoding.
print("\n=== TEXT SAMPLES FROM TRAIN PDFs ===")
for index in range(len(train_pdfs)):
    row = audit_table.iloc[index]
    print(row["file_name"], f"({row['page_count']} pages):", row["sample_text"])

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Total train PDFs: 0 | Total test PDFs: 0


KeyError: 'page_count'

Cell 3b: Check what is actually inside train_pdfs and test_pdfs
cell 3a installed yMuPDF (fitz) but couldn't
top-level file search returned zero PDFs

In [ ]:
def inspect_subfolder(folder_path, max_items):
    """
    Input: Path to a folder and maximum number of items to display.
    Output: Prints immediate contents and counts all files found recursively.
    """
    print("====================================")
    print("Inspecting:", folder_path)

    # Show immediate items so we can see if there are subfolders or nested zips.
    top_items = sorted(folder_path.iterdir())
    print("Immediate items count:", len(top_items))
    for item in top_items[:max_items]:
        item_type = "FOLDER" if item.is_dir() else "FILE"
        print(f" - [{item_type}] {item.name}")

    # Search all nested folders to find where the actual PDFs live.
    all_pdfs = sorted(folder_path.rglob("*.pdf"))
    print("Total .pdf files found recursively:", len(all_pdfs))
    if len(all_pdfs) > 0:
        print("Example PDF path:", all_pdfs[0])


inspect_subfolder(CONFIG["data_dir"] / "train_pdfs", max_items=10)
inspect_subfolder(CONFIG["data_dir"] / "test_pdfs", max_items=10)

Inspecting: /content/lava_data/train_pdfs
Immediate items count: 1
 - [FOLDER] train_pdfs
Total .pdf files found recursively: 5
Example PDF path: /content/lava_data/train_pdfs/train_pdfs/j_0030.pdf
Inspecting: /content/lava_data/test_pdfs
Immediate items count: 1
 - [FOLDER] test_pdfs
Total .pdf files found recursively: 200
Example PDF path: /content/lava_data/test_pdfs/test_pdfs/j_0000.pdf


Cell 4: Run the PDF audit (OCR & VLM gates)

In [ ]:
import random
import pymupdf
import pandas as pd

# Save the actual nested PDF folders into CONFIG (copy these two lines to Cell 1 later).
CONFIG["train_pdf_dir"] = CONFIG["data_dir"] / "train_pdfs" / "train_pdfs"
CONFIG["test_pdf_dir"] = CONFIG["data_dir"] / "test_pdfs" / "test_pdfs"


def audit_single_pdf(pdf_path, min_chars):
    """
    Input: Path to one PDF file and minimum character threshold.
    Output: Dictionary with page count, low-text count, visual count, and a text sample.
    """
    document = pymupdf.open(pdf_path)
    low_text_pages = 0
    visual_pages = 0
    sample_text = ""

    for page in document:
        page_text = page.get_text("text").strip()
        if len(page_text) < min_chars:
            low_text_pages = low_text_pages + 1
        # Images or many vector drawings usually mean charts, diagrams, or complex tables.
        if len(page.get_images()) > 0 or len(page.get_drawings()) > 20:
            visual_pages = visual_pages + 1
        if sample_text == "" and len(page_text) >= min_chars:
            sample_text = page_text[:120].replace("\n", " ")

    page_count = len(document)
    document.close()
    return {
        "file_name": pdf_path.name,
        "page_count": page_count,
        "low_text_pages": low_text_pages,
        "visual_pages": visual_pages,
        "sample_text": sample_text,
    }


def print_audit_summary(audit_table, min_chars):
    """
    Input: DataFrame of per-PDF audit results and minimum character threshold.
    Output: Prints page counts and percentages for the OCR and VLM gates.
    """
    total_pages = audit_table["page_count"].sum()
    low_text_total = audit_table["low_text_pages"].sum()
    visual_total = audit_table["visual_pages"].sum()

    low_text_pct = (low_text_total / max(1, total_pages)) * 100
    visual_pct = (visual_total / max(1, total_pages)) * 100

    print("Audited PDFs:", len(audit_table), "| Total pages:", total_pages)
    print(
        "Pages per PDF -> min:", audit_table["page_count"].min(),
        "| median:", audit_table["page_count"].median(),
        "| max:", audit_table["page_count"].max(),
    )
    print(f"Low-text pages (<{min_chars} chars, OCR gate): {low_text_total} ({low_text_pct:.1f}%)")
    print(f"Visual pages (images/drawings, VLM gate): {visual_total} ({visual_pct:.1f}%)")


train_pdfs = sorted(CONFIG["train_pdf_dir"].glob("*.pdf"))
test_pdfs = sorted(CONFIG["test_pdf_dir"].glob("*.pdf"))
print("Found train PDFs:", len(train_pdfs), "| Found test PDFs:", len(test_pdfs))

# Fixed seed makes the 25-PDF sample identical every time you run this cell.
random.seed(42)
sample_test_pdfs = random.sample(test_pdfs, min(25, len(test_pdfs)))

rows = []
for pdf_path in train_pdfs + sample_test_pdfs:
    rows.append(audit_single_pdf(pdf_path, CONFIG["min_text_chars"]))

audit_table = pd.DataFrame(rows)
print_audit_summary(audit_table, CONFIG["min_text_chars"])

print("\n=== TEXT SAMPLES FROM TRAIN PDFs (CHECK FOR GARBLED TEXT) ===")
for index in range(len(train_pdfs)):
    row = audit_table.iloc[index]
    print(row["file_name"], f"({row['page_count']} pages):", row["sample_text"])

Found train PDFs: 5 | Found test PDFs: 200
Audited PDFs: 30 | Total pages: 586
Pages per PDF -> min: 1 | median: 18.0 | max: 62
Low-text pages (<50 chars, OCR gate): 84 (14.3%)
Visual pages (images/drawings, VLM gate): 464 (79.2%)

=== TEXT SAMPLES FROM TRAIN PDFs (CHECK FOR GARBLED TEXT) ===
j_0030.pdf (18 pages): 第2節 デジタルデータ活用の現状と課題 1 日本におけるデジタルデータ活用の現状 これまで見てきたように、デジタルデータの流通量は増大しており、また国境を越えたデータのやりとりも増加 傾向にある。 企業によるこのようなデータの活用はこれまで
j_0057.pdf (4 pages): 08 愛知県 ■建築主  ／愛知県 ■所在地  ／〒444-0802 愛知県岡崎市美合町並松1-51 ■敷地面積 ／17,531㎡ ■建築面積 ／3,623㎡ ■延床面積 ／7,992㎡ ■構造・規模 ／鉄筋コンクリート造（一部 鉄骨造） 
j_0060.pdf (15 pages): 革新的モデルと観測・室内実験による 有機エアロゾルの生成機構と起源の解明 国立環境研究所 群馬県衛生環境研究所 高崎経済大学 名古屋市環境科学調査センター 東京都環境科学研究所 環境研究総合推進費5-1801 事後評価ヒアリング資料 課題番
j_0158.pdf (21 pages): 資料３ ◇新たな時代にはばたく“ふくろいスタイル”（Ｐ1） ◇主要プロジェクト １ 幼小中一貫教育の推進 （Ｐ２） ２ ＩＣＴを活かした教育推進事業 （Ｐ３） ３ 待機児童対策事業 （Ｐ４） ４ 袋井型英語教育推進事業 （Ｐ５） ５ ３Da
v_0104.pdf (16 pages): THẺ TÍN DỤNG QUỐC TẾ WOORI BANK - DANH SÁCH MCC TỪNG LĨNH VỰC  VÀ DANH SÁCH MCC LOẠI TRỪ  WOORI BANK VISA CR

Cell 4: Run the PDF audit (OCR & VLM gates)

In [ ]:
import ast


def inspect_low_text_pages(pdf_paths, min_chars):
    """
    Input: List of PDF paths and minimum character threshold.
    Output: Prints how many low-text pages contain images and shows 10 examples.
    """
    low_text_with_images = 0
    low_text_no_images = 0
    examples = []

    for pdf_path in pdf_paths:
        document = pymupdf.open(pdf_path)
        for page_index in range(len(document)):
            page = document[page_index]
            page_text = page.get_text("text").strip()
            if len(page_text) < min_chars:
                image_count = len(page.get_images())
                if image_count > 0:
                    low_text_with_images = low_text_with_images + 1
                else:
                    low_text_no_images = low_text_no_images + 1
                if len(examples) < 10:
                    examples.append((pdf_path.name, page_index + 1, len(page_text), image_count, page_text[:40]))
        document.close()

    print("Low-text pages WITH images (likely scanned or figures):", low_text_with_images)
    print("Low-text pages WITHOUT images (likely blank or title slides):", low_text_no_images)
    print("\nFirst 10 low-text page examples (file, 1-based page, chars, images, text):")
    for item in examples:
        print(" -", item)


def check_train_evidence_pages(train_table, train_pdf_dir, min_chars):
    """
    Input: Training DataFrame, path to train PDFs, and minimum character threshold.
    Output: Prints how many ground-truth evidence pages are low-text.
    """
    total_evidence = 0
    low_text_evidence = 0

    for row_index in range(len(train_table)):
        row = train_table.iloc[row_index]
        pdf_path = train_pdf_dir / f"{row['file_id']}.pdf"
        page_numbers = ast.literal_eval(row["evidence_page_number"])
        document = pymupdf.open(pdf_path)

        # Evidence page numbers are 1-indexed, so subtract 1 for PyMuPDF.
        for page_num in page_numbers:
            total_evidence = total_evidence + 1
            page_text = document[page_num - 1].get_text("text").strip()
            if len(page_text) < min_chars:
                low_text_evidence = low_text_evidence + 1
                print("Low-text train evidence:", row["id"], row["file_id"], "page", page_num, "chars:", len(page_text))
        document.close()

    print(f"\nTrain evidence pages checked: {total_evidence} | Low-text (<{min_chars} chars): {low_text_evidence}")


train_table = pd.read_csv(CONFIG["data_dir"] / "train.csv")
inspect_low_text_pages(train_pdfs + sample_test_pdfs, CONFIG["min_text_chars"])
check_train_evidence_pages(train_table, CONFIG["train_pdf_dir"], CONFIG["min_text_chars"])

Low-text pages WITH images (likely scanned or figures): 77
Low-text pages WITHOUT images (likely blank or title slides): 7

First 10 low-text page examples (file, 1-based page, chars, images, text):
 - ('j_0028.pdf', 1, 26, 1, '平成24年2月 \n総務省 \n \n \n地方財政関係資料')
 - ('j_0028.pdf', 18, 43, 3, '地方債の元利償還金に対する交付税措置 \n（いわゆる事業費補正）の見直し（概要） ')
 - ('v_0064.pdf', 34, 0, 1, '')
 - ('j_0061.pdf', 1, 7, 0, '人材育成\n74')
 - ('j_0149.pdf', 1, 0, 11, '')
 - ('j_0149.pdf', 2, 0, 31, '')
 - ('j_0149.pdf', 3, 0, 69, '')
 - ('j_0149.pdf', 4, 0, 12, '')
 - ('j_0149.pdf', 5, 0, 14, '')
 - ('j_0149.pdf', 6, 0, 48, '')

Train evidence pages checked: 27 | Low-text (<50 chars): 0
